In [2]:
import sys, io, warnings, time, json
from pathlib import Path
from contextlib import redirect_stdout
from itertools import product, combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

from run_simulation import run_simulation
from utils.metrics import (
    compute_metrics, build_result_row, save_result_row,
    aggregate_config, load_all_aggregated,
)

# ── Paden ──────────────────────────────────────────────────────────────────
RESULTS_DIR = Path.cwd().parent / "results" / "Timing_Strategies_25/05"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results dir: {RESULTS_DIR.resolve()}")

Results dir: /Users/ddw/Desktop/Rescheduling/results/Timing_Strategies_25/05


In [3]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                    BATCH-INSTELLINGEN (AANPASSEN!)                  ║
# ╚══════════════════════════════════════════════════════════════════════╝
TARGET_SEEDS = 50          # Vul elke config tot dit aantal seeds (0..TARGET_SEEDS-1)

# ── Welke trigger-types meenemen? ──────────────────────────────────────────
INCLUDE_BASELINE     = True   # FCFS only — 1 config (periodic_freq=100_000)
INCLUDE_PERIODIC     = True   # 7 configs (verschillende periodic_freq)
INCLUDE_EVENT_DRIVEN = True    # 39 configs (5 ed_rf × cf × 3 tc)
INCLUDE_HYBRID       = True    # 66 configs (22 (ed_rf,cf,prf) × 3 tc)

# ── Welke priorities meenemen? ─────────────────────────────────────────────
INCLUDE_NO_PRIORITY = True
INCLUDE_STATIC      = True
INCLUDE_DYNAMIC     = True

# ── Per-seed sim-output (df met trein-niveau detail) als parquet bewaren? ──
# True  = parquets per seed komen in cfg_dir/<auto_naam>/seed_X.parquet
#         (handig voor latere diepteanalyses zonder reruns)
# False = alleen raw_runs.csv met geaggregeerde metrics (lichter op disk)
SAVE_PER_SEED_PARQUET = True

# ── Vaste simulatieparameters (overgenomen van Calibrate_Timing) ───────────
N_FREIGHT          = 182      # ≈15% freight share
FREIGHT_PCT        = 0.15
GAMMA              = 120      # s — dynamic_threshold (uit calibrate_gamma)
MC_DELAY_PER_TRAIN = 120      # s — Monte-Carlo delay per actieve trein
SOLVER_TIMEOUT     = 60       # s
PHASE              = 1

# ── Priority-gewichten (overgenomen van Fase1) ─────────────────────────────
WEIGHT_PASS_PRIO = 1     # weight_passenger als priority actief (static/dynamic)
WEIGHT_FREIGHT_PRIO = 1  # weight_freight als priority actief
UPGRADE_WEIGHT      = 1     # extra penalty als delay >= gamma (alleen bij dynamic)
subtype_weights = {
      "EURST":   10,   # Internationaal HSL — slot agreements, hoge boetes
      "ICE":     10,   # Idem internationale HSL
      "INT":      5,   # Internationaal (regulier)
      "IC":       3,   # Domestic backbone
      "S":        3,   # Veel pendelaars in piek — Brusselse corridor!
      "L":        2,   # Lokaal
      "freight":  1,   # Basislijn
      }

print(f"TARGET_SEEDS     = {TARGET_SEEDS}")
print(f"Triggers actief  : baseline={INCLUDE_BASELINE}, periodic={INCLUDE_PERIODIC}, "
      f"event_driven={INCLUDE_EVENT_DRIVEN}, hybrid={INCLUDE_HYBRID}")
print(f"Priorities actief: no_priority={INCLUDE_NO_PRIORITY}, "
      f"static={INCLUDE_STATIC}, dynamic={INCLUDE_DYNAMIC}")
print(f"Save parquets    = {SAVE_PER_SEED_PARQUET}")
print(f"GAMMA            = {GAMMA}s  |  MC_DELAY = {MC_DELAY_PER_TRAIN}s")
configs = []
# -----------------------------------------------------------------------
# Baseline — FCFS only, solver never fires
# -----------------------------------------------------------------------
configs.append({
    'trigger_strategy':     'periodic',
    'periodic_freq':        100_000,
    'event_driven_freq':    900,
    'controller_freq':      300,
    'threshold_confidence': 0.6,
})
# -----------------------------------------------------------------------
# Periodic — 7 values
# -----------------------------------------------------------------------
for prf in [450, 900, 1800, 3600]:
    configs.append({
        'trigger_strategy':     'periodic',
        'periodic_freq':        prf,
        'event_driven_freq':    900,
        'controller_freq':      300,
        'threshold_confidence': 0.6,
    })
# -----------------------------------------------------------------------
# # Event-Driven — inclusief nieuwe edf=600 configs
# # -----------------------------------------------------------------------
ed_rf_cf_map = {
    600:  [300, 600],
    900:  [450, 900],
    1800: [450, 900, 1800],
    3600: [900, 1800, 3600],
    5400: [1800, 3600, 5400],
}
tc_values = [0.4, 0.8]
for ed_rf, cf_list in ed_rf_cf_map.items():
    for cf in cf_list:
        for tc in tc_values:
            configs.append({
                'trigger_strategy':     'event_driven',
                'event_driven_freq':    ed_rf,
                'controller_freq':      cf,
                'threshold_confidence': tc,
                'periodic_freq':        100_000,
            })
# -----------------------------------------------------------------------
# Hybrid — bestaande + nieuwe met ed_rf=900 en ed_rf=600
# -----------------------------------------------------------------------
hybrid_combos = [
    # --- bestaand ---
    (1800, 900,  3600), 
    (1800, 900,  5400), 
    (1800, 900,  7200), 
    (3600, 900,  5400), (3600, 1800, 5400), 
    (3600, 900,  7200), (3600, 1800, 7200), 
    (5400, 900,  7200), (5400, 1800, 7200),
    # --- nieuw: ed_rf=900 ---
    (900, 450, 1800), (900, 450, 3600),
]
for ed_rf, cf, prf in hybrid_combos:
    for tc in tc_values:
        configs.append({
            'trigger_strategy':     'hybrid',
            'event_driven_freq':    ed_rf,
            'controller_freq':      cf,
            'periodic_freq':        prf,
            'threshold_confidence': tc,
        })
# -----------------------------------------------------------------------
# Tellen
# -----------------------------------------------------------------------
n_baseline     = sum(1 for c in configs if c['trigger_strategy'] == 'periodic' and c['periodic_freq'] == 100_000)
n_periodic     = sum(1 for c in configs if c['trigger_strategy'] == 'periodic' and c['periodic_freq'] != 100_000)
n_event_driven = sum(1 for c in configs if c['trigger_strategy'] == 'event_driven')
n_hybrid       = sum(1 for c in configs if c['trigger_strategy'] == 'hybrid')
print(f'Total configurations: {len(configs)}')
print(f'  Baseline:     {n_baseline}')
print(f'  Periodic:     {n_periodic}')
print(f'  Event-Driven: {n_event_driven}  (+6 nieuw: edf=600 × 2 cf × 3 tc)')
print(f'  Hybrid:       {n_hybrid}  (+12 nieuw: ed_rf=900/600 × 4 combos × 3 tc)')
print(f'\nTotal runs: {len(configs)} x {TARGET_SEEDS} = {len(configs) * TARGET_SEEDS}')
# ── Priority-configuraties (uit Fase1_Experiment.ipynb) ───────────────────
priority_configs = {}

if INCLUDE_NO_PRIORITY:
    priority_configs["no_priority"] = dict(
        objective_strategy = "static",
        weight_passenger   = 1,
        weight_freight     = 1,
        upgrade_weight     = UPGRADE_WEIGHT,
        dynamic_threshold  = GAMMA,
    )
# if INCLUDE_STATIC:
#     priority_configs["static"] = dict(
#         objective_strategy = "static",
#         weight_passenger   = WEIGHT_PASS_PRIO,
#         weight_freight     = WEIGHT_FREIGHT_PRIO,
#         upgrade_weight     = UPGRADE_WEIGHT,
#         subtype_weights = subtype_weights,
#         dynamic_threshold  = GAMMA,
#     )
# if INCLUDE_DYNAMIC:
#     priority_configs["dynamic"] = dict(
#         objective_strategy = "dynamic",
#         weight_passenger   = WEIGHT_PASS_PRIO,
#         weight_freight     = WEIGHT_FREIGHT_PRIO,
#         upgrade_weight     = UPGRADE_WEIGHT,
#         dynamic_threshold  = GAMMA,
#     )

PRIORITY_ORDER = [p for p in ["no_priority", "static", "dynamic"] if p in priority_configs]
TRIGGER_ORDER  = [t for t in ["periodic", "event_driven", "hybrid"]
                  if any(c["trigger_strategy"] == t for c in configs)]

print(f"Priorities: {PRIORITY_ORDER}")
print(f"Triggers:   {TRIGGER_ORDER}")

# ── Volledig grid: Cartesiaans product ────────────────────────────────────
CONFIGS = list(product(configs, PRIORITY_ORDER))

print(f"\nTotaal configs (timing × priority): {len(CONFIGS)}")
print(f"Totaal runs voor TARGET_SEEDS={TARGET_SEEDS}: {len(CONFIGS) * TARGET_SEEDS}")
def load_all_raw() -> pd.DataFrame:
    """Laadt alle raw_runs.csv bestanden uit RESULTS_DIR in één DataFrame."""
    frames = []
    for cfg_dir in sorted(RESULTS_DIR.iterdir()):
        if not cfg_dir.is_dir():
            continue
        csv_path = cfg_dir / "raw_runs.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["config"] = cfg_dir.name
            frames.append(df)
    if not frames:
        print("Geen resultaten gevonden.")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


raw = load_all_raw()
print(f"Geladen: {len(raw)} rijen uit {raw['config'].nunique()} configs")

# Filter deadlocks uit voor performantie-analyses (maar bewaar voor overzicht)
if "deadlock" in raw.columns:
    n_deadlock = raw["deadlock"].fillna(False).astype(bool).sum()
    print(f"Deadlocks: {n_deadlock} runs (worden uitgesloten van metric-analyses)")
    raw_all = raw.copy()
    raw = raw[~raw["deadlock"].fillna(False).astype(bool)].copy()
else:
    raw_all = raw.copy()

# Canonieke ordering
raw["trigger"]  = pd.Categorical(raw["strategy_type"], categories=TRIGGER_ORDER, ordered=True)
raw["priority"] = pd.Categorical(raw["priority"],      categories=PRIORITY_ORDER, ordered=True)

print(f"\nNa filtering deadlocks: {len(raw)} rijen voor analyse")
def load_all_raw() -> pd.DataFrame:
    """Laadt alle raw_runs.csv bestanden uit RESULTS_DIR in één DataFrame."""
    frames = []
    for cfg_dir in sorted(RESULTS_DIR.iterdir()):
        if not cfg_dir.is_dir():
            continue
        csv_path = cfg_dir / "raw_runs.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["config"] = cfg_dir.name
            frames.append(df)
    if not frames:
        print("Geen resultaten gevonden.")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


raw = load_all_raw()
print(f"Geladen: {len(raw)} rijen uit {raw['config'].nunique()} configs")

# Filter deadlocks uit voor performantie-analyses (maar bewaar voor overzicht)
if "deadlock" in raw.columns:
    n_deadlock = raw["deadlock"].fillna(False).astype(bool).sum()
    print(f"Deadlocks: {n_deadlock} runs (worden uitgesloten van metric-analyses)")
    raw_all = raw.copy()
    raw = raw[~raw["deadlock"].fillna(False).astype(bool)].copy()
else:
    raw_all = raw.copy()

# Canonieke ordering
raw["trigger"]  = pd.Categorical(raw["strategy_type"], categories=TRIGGER_ORDER, ordered=True)
raw["priority"] = pd.Categorical(raw["priority"],      categories=PRIORITY_ORDER, ordered=True)

print(f"\nNa filtering deadlocks: {len(raw)} rijen voor analyse")


TARGET_SEEDS     = 50
Triggers actief  : baseline=True, periodic=True, event_driven=True, hybrid=True
Priorities actief: no_priority=True, static=True, dynamic=True
Save parquets    = True
GAMMA            = 120s  |  MC_DELAY = 120s
Total configurations: 53
  Baseline:     1
  Periodic:     4
  Event-Driven: 26  (+6 nieuw: edf=600 × 2 cf × 3 tc)
  Hybrid:       22  (+12 nieuw: ed_rf=900/600 × 4 combos × 3 tc)

Total runs: 53 x 50 = 2650
Priorities: ['no_priority']
Triggers:   ['periodic', 'event_driven', 'hybrid']

Totaal configs (timing × priority): 53
Totaal runs voor TARGET_SEEDS=50: 2650
Geladen: 2650 rijen uit 53 configs
Deadlocks: 46 runs (worden uitgesloten van metric-analyses)

Na filtering deadlocks: 2604 rijen voor analyse
Geladen: 2650 rijen uit 53 configs
Deadlocks: 46 runs (worden uitgesloten van metric-analyses)

Na filtering deadlocks: 2604 rijen voor analyse
